# Module 09 — Topic 02: Scaling Laws in Natural Hazards
**Completed exercises: Seismic catalogue preparation (I) and (II)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

# Exercise I — Catalogue preparation
## 1) Load the official catalogue and remove duplicate events
The ID follows the lecture by concatenating Date, Time, Longitude, and Latitude.

In [ ]:
url=('https://raw.githubusercontent.com/Digihaz/digihaz-course-materials/dev/'
     'module_09_big_data_scaling_laws/topic_02/notebooks/catalogue_exercise.csv')
data=pd.read_csv(url,sep=';',skipinitialspace=True)
data.columns=data.columns.str.strip()
for c in data.select_dtypes(include='object').columns:
    data[c]=data[c].str.strip()
n_original=len(data)
data['ID']=data[['Date','Time','Longitude','Latitude']].astype(str).agg(''.join,axis=1)
duplicates=data[data.duplicated('ID',keep=False)].copy()
data=data.drop_duplicates('ID',keep='first').copy()
print('Original events:',n_original)
print('Duplicated rows removed:',n_original-len(data))
print('Rows after duplicate removal:',len(data))
display(duplicates.head(10))

**Reproduced result from the official dataset:** 48,121 original rows; **21 duplicate rows removed**; 48,100 rows remain.

## 2) Convert Roman-numeral intensity to decimal

In [ ]:
intensity_map={'I':1.0,'I-II':1.5,'II':2.0,'II-III':2.5,'III':3.0,'III-IV':3.5,'IV':4.0,'IV-V':4.5,'V':5.0,'V-VI':5.5,'VI':6.0,'VI-VII':6.5,'VII':7.0,'VII-VIII':7.5,'VIII':8.0,'VIII-IX':8.5,'IX':9.0,'IX-X':9.5,'X':10.0,'X-XI':10.5,'XI':11.0,'XI-XII':11.5,'XII':12.0}
data['I_num']=data['Intensity'].map(intensity_map).fillna(-1.0)
display(data[['Intensity','I_num']].drop_duplicates().sort_values('I_num').head(30))

## 3) Homogenise magnitude/intensity values to Mw

In [ ]:
data['Magnitude']=pd.to_numeric(data['Magnitude'],errors='coerce')
data['Magnitude_type']=pd.to_numeric(data['Magnitude_type'],errors='coerce')
c1=data['Magnitude_type'].isna(); c2=data['Intensity'].isna() | (data['Intensity']=='')
data.loc[c1 & c2,'Magnitude_type']=0
data.loc[c1 & ~c2,'Magnitude_type']=1

def mw_converter(df):
    eqs={1:{'a':1.6560,'b':0.545,'min':3.0,'max':9.5,'col':'I_num'},
         2:{'a':0.2900,'b':0.973,'min':3.1,'max':7.3,'col':'Magnitude'},
         3:{'a':-1.528,'b':1.213,'min':3.7,'max':6.3,'col':'Magnitude'},
         4:{'a':0.6760,'b':0.836,'min':3.0,'max':5.1,'col':'Magnitude'}}
    mw=np.full(len(df),-9999.0)
    for t,e in eqs.items():
        mask=(df['Magnitude_type'].eq(t) & df[e['col']].ge(e['min']) & df[e['col']].le(e['max']))
        mw[mask]=e['a']+e['b']*df.loc[mask,e['col']]
    mask=df['Magnitude_type']>=5
    mw[mask]=df.loc[mask,'Magnitude']
    df['mw']=mw
    return df

data=mw_converter(data)
data=data[data['mw']>-9999].copy()
print('Events with usable homogenised Mw:',len(data))

**Reproduced result:** **3,450 events** have a usable homogenised Mw after applying the lecture equations and stated validity ranges.

## 4) Compute DecimalYear, plot Mw vs time, and export the processed catalogue

In [ ]:
dt=pd.to_datetime(data['Date']+' '+data['Time'],format='%d/%m/%Y %H:%M:%S',errors='coerce')
year=dt.dt.year
start=pd.to_datetime(year.astype('Int64').astype(str)+'-01-01',errors='coerce')
end=pd.to_datetime((year+1).astype('Int64').astype(str)+'-01-01',errors='coerce')
data['DecimalYear']=year+(dt-start)/(end-start)
data=data.dropna(subset=['DecimalYear']).copy()
plt.figure(figsize=(9,5)); plt.scatter(data['DecimalYear'],data['mw'],s=8)
plt.xlabel('Time [year]'); plt.ylabel('Magnitude [Mw]'); plt.title('Southern Spain Seismic Catalogue'); plt.show()
data.to_csv('catalogue_topic02_preprocessed.csv',index=False)

### Interpretation
The catalogue has strong observational incompleteness through time: pre-instrumental years contain far fewer recorded earthquakes, while the instrumental era contains many more small events. This mainly reflects improved detection and network coverage rather than an equivalent physical increase in seismicity. Clustering after larger earthquakes is also visible, motivating declustering before estimating background seismicity.

**Depth check:** 95 of the 3,450 usable events are deeper than 50 km, about **2.75%**, which is below the lecture's approximate 5% criterion for considering them negligible.

# Exercise II — Seismic parameters and declustering
Following the lecture, use **DecimalYear ≥ 1970** and **Mw ≥ 3.0**.

In [ ]:
analysis=data[(data['DecimalYear']>=1970)&(data['mw']>=3.0)].copy().reset_index(drop=True)
print('Analysis events:',len(analysis))
plt.figure(figsize=(9,5)); plt.scatter(analysis['DecimalYear'],analysis['mw'],s=8)
plt.xlabel('Time [year]'); plt.ylabel('Magnitude [Mw]'); plt.title('Analysis catalogue (1970+, Mw>=3)'); plt.show()
bins=np.arange(round(analysis.mw.min(),1),round(analysis.mw.max(),1)+0.2,0.1)
counts,edges=np.histogram(analysis.mw,bins=bins); inv=np.cumsum(counts[::-1])[::-1]; mask=inv>0
plt.figure(figsize=(7,5)); plt.scatter(edges[:-1][mask],np.log10(inv[mask]),s=12)
plt.xlabel('Magnitude [Mw]'); plt.ylabel(r'$log_{10}(N_{m\geq M})$'); plt.title('Inverse cumulative frequency-magnitude distribution'); plt.show()

## 3) Compute completeness magnitude and Gutenberg–Richter parameters

In [ ]:
def _inclusive_arange(start,stop,step):
    n=int(round((stop-start)/step)); return start+step*np.arange(n+1,dtype=float) if n>=0 else np.array([])
def calc_bmemag(mags,bin_interval=0.1):
    mags=np.asarray(mags,float); mags=mags[np.isfinite(mags)]; n=len(mags)
    mn=float(np.min(mags)); mean=float(np.mean(mags)); denom=mean+0.5*bin_interval-mn
    b=(1.0/denom)*math.log10(math.e); var=float(np.var(mags-mean,ddof=1)/n); bstd=2.30*math.sqrt(max(0,var))*b*b
    a=math.log10(n)+b*mn; return b,bstd,a
def _do_calc(mags,bin_interval,mc):
    b,_,_=calc_bmemag(mags,bin_interval); half=0.5*bin_interval; vmag=_inclusive_arange(mc,15,bin_interval)
    vnum=np.round(10**(math.log10(len(mags))-b*(vmag-mc)))
    ed=np.concatenate([vmag-half,[vmag[-1]+half]]); h=np.histogram(mags,ed)[0].astype(float); c=np.cumsum(h[::-1])[::-1]
    return float(np.sum(np.abs(c-vnum))/np.sum(c)*100)
def _max_curv(mags,bin_interval):
    half=0.5*bin_interval; c=_inclusive_arange(-2,6,bin_interval); ed=np.concatenate([c-half,[c[-1]+half]])
    h=np.histogram(mags,ed)[0]; return float(c[np.where(h==np.max(h))[0][-1]])
def calc_mc_best(magnitudes,bin_interval=0.1):
    mags=np.sort(np.asarray(magnitudes,float)); mags=mags[np.isfinite(mags)]; seed=_max_curv(mags,bin_interval); half=0.5*bin_interval
    centers=_inclusive_arange(seed-0.9,seed+1.5,bin_interval); desc=mags[::-1]; res=np.full(centers.shape,np.nan)
    for i,mc in enumerate(centers):
        n=int(np.sum(desc>(mc-half)))
        if n>=25: res[i]=_do_calc(desc[:n],bin_interval,float(mc))
    def first(thr):
        idx=np.where(res<thr)[0]; return float(centers[idx[0]]) if len(idx) else np.nan
    best=np.nan
    for thr in (10,15,20,25):
        v=first(thr)
        if np.isfinite(v): best=v; break
    return best,first(5),first(10)
def seism_params(mags,mc,deltaM=0.1):
    mags=np.asarray(mags,float); b=np.log10(np.e)/(np.mean(mags)+0.5*deltaM-mc); a=np.log10(len(mags))+b*mc; return a,b
mc=calc_mc_best(analysis.mw); a0,b0=seism_params(analysis.mw,mc[0])
print('Mc(best, Mc95, Mc90)=',mc); print('a=',a0,'b=',b0)

**Reproduced before declustering:** N = **1,426**, Mc ≈ **3.3**, a ≈ **7.9024**, b ≈ **1.4389**.

## 4) Decluster using Gardner–Knopoff and Gruenthal window methods

In [ ]:
def haversine_vector(lon,lat,lon0,lat0):
    R=6371.0; lon=np.radians(lon); lat=np.radians(lat); lon0=math.radians(lon0); lat0=math.radians(lat0)
    dlon=lon-lon0; dlat=lat-lat0; a=np.sin(dlat/2)**2+np.cos(lat)*math.cos(lat0)*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a))
def spatial_GK(m): return 10**(0.1238*m+0.983)
def time_GK(m,days=60.0):
    m=np.asarray(m,float); out=10**(0.032*m+2.7389)/days; mask=m<6.5; out[mask]=10**(0.5409*m[mask]-0.547)/days; return out
def spatial_G(m): return np.exp(1.77+np.sqrt(0.037+1.02*m))
def time_G(m,days=60.0):
    m=np.asarray(m,float); out=10**(2.8+0.024*m)/days; mask=m<6.5; out[mask]=np.abs(np.exp(-3.95+np.sqrt(0.62+17.32*m[mask]))/days); return out
def decluster_window(df,method='GK'):
    work=df.reset_index(drop=True).copy(); order=np.argsort(work.mw.to_numpy())[::-1]
    mw=work.mw.to_numpy()[order]; yr=work.DecimalYear.to_numpy()[order]; lon=work.Longitude.astype(float).to_numpy()[order]; lat=work.Latitude.astype(float).to_numpy()[order]
    sw=spatial_GK(mw) if method=='GK' else spatial_G(mw); tw=time_GK(mw) if method=='GK' else time_G(mw)
    n=len(work); vcl=np.zeros(n,int); flag=np.zeros(n,int); cid=0
    for i in range(n-1):
        if vcl[i]!=0: continue
        dt=yr-yr[i]; cand=(vcl==0)&(dt>=-tw[i])&(dt<=tw[i]); idx=np.where(cand)[0]; inside=np.zeros(n,bool)
        if len(idx):
            d=haversine_vector(lon[idx],lat[idx],lon[i],lat[i]); inside[idx[d<=sw[i]]]=True
        tmp=inside.copy(); tmp[i]=False
        if np.any(tmp):
            cid+=1; vcl[inside]=cid; flag[inside]=1; flag[tmp&(dt<0)]=-1; flag[i]=0
    back=np.argsort(order); work['Flag']=flag[back]; work['Cluster_label']=vcl[back]; return work
gk=decluster_window(analysis,'GK'); gr=decluster_window(analysis,'G')

## 5) Compare declustering results and recompute G–R parameters

In [ ]:
def summarize(df,name):
    rows=[]
    for f,label in [(0,'Main/background'),(1,'Aftershock'),(-1,'Foreshock')]:
        n=int((df.Flag==f).sum()); rows.append([name,label,n,100*n/len(df)])
    main=df[df.Flag==0]; mc=calc_mc_best(main.mw); a,b=seism_params(main.mw,mc[0]); return rows,{'Method':name,'N_background':len(main),'Mc':mc[0],'a':a,'b':b}
r1,s1=summarize(gk,'Gardner-Knopoff'); r2,s2=summarize(gr,'Gruenthal')
display(pd.DataFrame(r1+r2,columns=['Method','Class','Count','Percent']).round(3))
display(pd.DataFrame([s1,s2]).round(4))
gk.to_csv('catalogue_topic02_declustered_GK.csv',index=False); gr.to_csv('catalogue_topic02_declustered_Gruenthal.csv',index=False)

### Reproduced comparison
| Method | Main/background | Aftershocks | Foreshocks | Mc | a | b |
|---|---:|---:|---:|---:|---:|---:|
| Gardner–Knopoff | 648 (45.44%) | 458 (32.12%) | 320 (22.44%) | 3.3 | 6.7996 | 1.2085 |
| Gruenthal | 380 (26.65%) | 645 (45.23%) | 401 (28.12%) | 3.3 | 5.9184 | 1.0117 |

The methods identify different proportions of clustered seismicity, so declustering is method-dependent. In both cases the b-value decreases relative to the non-declustered catalogue (≈1.4389), changing the inferred small-to-large earthquake ratio. This sensitivity should be reported whenever the catalogue is used for hazard analysis.